# Mooring Field Detection — Cape Cod + Florida (one multi-zip dataset)

Detects mooring fields on **pre-fetched** tiles. No Google Maps fetch. No Groq.

## Kaggle settings (required)

1. **Accelerator → GPU T4** (then **Restart session** if you just changed it)
2. **Internet → On**
3. **Add input** → attach your dataset that contains all `kaggle_scan_*.zip` files
4. **Run All**

This notebook finds **every** `kaggle_scan_*.zip` under `/kaggle/input` (including many zips in one dataset), runs detection on each, and writes one shared `scan_out/mooring_fields.db`.

Expect a long run if you attached all FL regions + Cape Cod (often 1–3+ hours on T4).

## After success (on your PC)

Download `scan_out/mooring_fields.db`, then:
```powershell
cd C:\Users\ishan\Downloads\MooringFieldDetection
python -m mooring_fields.cli import-scan --from-db path\to\mooring_fields.db --all
```
Hard-refresh the web map. Skip enrich for now.

In [ ]:
# Cell 1 — sparse clone + install
import subprocess, sys, shutil, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", URL, str(REPO)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO), "sparse-checkout", "set", "src", "config", "pyproject.toml", "README.md"],
    check=True,
)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"],
    check=True,
)

import torch
from mooring_fields.runtime import cuda_available

print("cuda:", cuda_available(), torch.cuda.get_device_name(0) if cuda_available() else None)
print("inputs:", sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else None)
assert cuda_available(), "Settings → Accelerator → GPU T4 → Restart session, then re-run"

In [ ]:
# Cell 2 — list EVERY payload zip (works with one dataset containing many zips)
from pathlib import Path

INPUT = Path("/kaggle/input")
assert INPUT.exists() and any(INPUT.iterdir()), "Add input → attach your multi-zip dataset"

# Prefer kaggle_scan_*.zip; fall back to any *.zip
zip_paths = sorted(INPUT.rglob("kaggle_scan_*.zip"))
if not zip_paths:
    zip_paths = sorted(INPUT.rglob("*.zip"))

# Drop duplicate Cape Cod payload if both names exist
names = {z.name for z in zip_paths}
if "kaggle_scan_CapeCod.zip" in names and "kaggle_scan_payload.zip" in names:
    zip_paths = [z for z in zip_paths if z.name != "kaggle_scan_payload.zip"]

assert zip_paths, (
    "No zip files under /kaggle/input. Your dataset must contain files like "
    "kaggle_scan_CapeCod.zip, kaggle_scan_FL_tampa_sw_p0.zip, …"
)

print(f"Found {len(zip_paths)} zip(s) to infer:")
for z in zip_paths:
    print(f"  - {z}  ({z.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# Cell 3 — extract one zip at a time → GPU detect → append into one DB → free disk
import json, os, sys, shutil, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.cli import scan_cmd
from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

out_dir = Path("/kaggle/working/scan_out")
out_dir.mkdir(parents=True, exist_ok=True)
db_path = out_dir / "mooring_fields.db"
extract_root = Path("/kaggle/working/payload_one")

reports = []
for i, zip_path in enumerate(zip_paths, 1):
    label = zip_path.stem  # e.g. kaggle_scan_FL_tampa_sw_p0
    print(f"\n======== [{i}/{len(zip_paths)}] {label} ========")

    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_root)

    kml = extract_root / "candidates.kml"
    if not kml.is_file():
        found = list(extract_root.rglob("candidates.kml"))
        assert found, f"{zip_path.name} has no candidates.kml"
        payload_dir = found[0].parent
    else:
        payload_dir = extract_root

    layout = materialize_kaggle_scan_input(
        payload_dir,
        work_dir=Path("/kaggle/working/scan_run") / label,
    )
    print(json.dumps({k: layout[k] for k in ("png_count", "weights")}, indent=2))
    assert layout["png_count"] > 0, layout
    assert layout["weights"], f"{label}: missing weights/best.pt"

    scan_cmd([
        "--kml", layout["kml"],
        "--skip-fetch",
        "--imagery-dir", layout["imagery_dir"],
        "--weights", layout["weights"],
        "--db", str(db_path),
        "--output-dir", str(out_dir / label),
    ])
    reports.append({"label": label, "png_count": layout["png_count"]})

    # Free disk before next zip
    shutil.rmtree(extract_root, ignore_errors=True)
    shutil.rmtree(Path("/kaggle/working/scan_run") / label, ignore_errors=True)

assert db_path.is_file()
print("\n========== SUCCESS ==========")
print("Download from Output:", db_path)
print("Batches run:")
for r in reports:
    print(" ", r)
print("On PC: python -m mooring_fields.cli import-scan --from-db <db> --all")

shutil.rmtree(REPO, ignore_errors=True)

## After the run

1. **Save Version** (or use the Output panel) → download `scan_out/mooring_fields.db`
2. On PC:
   ```powershell
   cd C:\Users\ishan\Downloads\MooringFieldDetection
   python -m mooring_fields.cli import-scan --from-db ~\Downloads\mooring_fields.db --all
   ```
3. Hard-refresh http://127.0.0.1:5173
4. Skip `enrich-all` until Groq works on your network